# ML S3 · Notebook 00 — The Session 3 Datasets

| | |
|---|---|
| **Session ID** | ML S3 · Notebook 00 of 05 |
| **Course position** | Machine Learning — Session 3 of 10 (Evaluation & Error Analysis) |
| **Block types** | Setup — run once, then leave it alone |
| **Prerequisites** | None. This notebook only generates data. |
| **Connects back to** | `ML_S1_00_dataset.ipynb` — the base housing generator is reproduced here verbatim |
| **Connects forward to** | Every other ML S3 notebook; the churn data continues into ML S4, S5 and S6 |
| **CEP linkage** | The churn dataset is a structural rehearsal for **Employee Turnover** (binary target, moderate imbalance, retention decision attached) |
| **Run requirements** | `pandas`, `numpy`. No internet. Runs in under 30 seconds. |
| **Checkpoint file** | `housing_extended.csv · subscription_churn.csv` |

> **Run this notebook once, first.** It writes two CSV files next to itself. Everything else in Session 3 reads those files.
>
> You do **not** need to understand the generator code. What you *do* need to read carefully is the **two data dictionaries** — especially the column called **"When is this known?"**. That column is the whole point of the first half of today's session, and it is the single most under-read column in professional data work.

## Why there are two datasets today

Session 3 is about **evaluation** — how you find out whether a model is actually any good. That question splits cleanly in two, and each half needs a different kind of data.

| Half of the session | Question being asked | Dataset |
|---|---|---|
| Notebooks 01–03 | Did I measure my **regression** model honestly? Did information sneak across the split? | `housing_extended.csv` |
| Notebooks 04–05 | For a **classification** model, what do precision and recall actually mean, and is model A genuinely better than model B? | `subscription_churn.csv` |

Housing predicts a **number** (price). You cannot build a confusion matrix on a number — there is nothing to confuse. Precision, recall, specificity and McNemar's test all need a **yes/no** prediction, so the second half of the session moves to a classification problem.

That is not busywork. It is the reason the two halves feel different, and it is worth saying out loud in the room.

---

## Part 1 — The extended housing dataset

### What changed since Session 1

In S1 and S2 you worked with `housing.csv`: 5,000 rows, nine columns. That was a **trimmed extract** — enough to learn regression on, deliberately kept simple.

The story for today is the one you will actually live through in a job:

> *The data team has sent the fuller extract. Same properties, but with the fields that were stripped out of the first sample — the transaction dates, the tax records, the development identifiers, and the original listing prices. "Use whatever's useful."*

This is exactly how new columns arrive in real work: in a bundle, undocumented beyond a one-line description, with no warning label on the dangerous ones. Some of the new columns will make your model look spectacular. Some of those are lying to you. Telling the difference is the skill this session builds.

In [1]:
# --- Setup ---
import numpy as np
import pandas as pd

SEED = 42
rng = np.random.default_rng(SEED)
N = 5000

### Step 1 — Rebuild the base housing data

This is the **identical generator from `ML_S1_00_dataset.ipynb`**, reproduced here so this notebook stands alone. If you still have `housing.csv` from Session 1, it will match this byte for byte — same seed, same 5,000 rows, same planted errors.

In [2]:
# Each metro maps to (price tier, price multiplier). Tier 1 = most expensive.
metros = {
    'San Francisco': (1, 1.90), 'New York': (1, 1.75), 'Boston': (1, 1.45),
    'Seattle': (2, 1.35), 'Austin': (2, 1.20), 'Denver': (2, 1.15), 'Chicago': (2, 1.05),
    'Phoenix': (3, 0.90), 'Columbus': (3, 0.80), 'Indianapolis': (3, 0.75),
}
metro_names = list(metros)
metro_probs = np.array([.11, .13, .08, .10, .11, .09, .12, .10, .08, .08])
metro_probs = metro_probs / metro_probs.sum()

metro           = rng.choice(metro_names, size=N, p=metro_probs)
location_tier   = np.array([metros[m][0] for m in metro])
city_multiplier = np.array([metros[m][1] for m in metro])

sqft          = rng.normal(1900, 650, N).clip(500, 6000).round(0)
bedrooms      = np.clip(np.round(sqft / 650 + rng.normal(0, 0.7, N)), 1, 7).astype(int)
bathrooms     = np.clip(np.round(bedrooms * 0.6 + rng.normal(0, 0.4, N)), 1, 5).astype(int)
house_age     = rng.integers(0, 80, N)
garage_spaces = rng.choice([0, 1, 2, 3], size=N, p=[.15, .45, .30, .10])
lot_size      = (sqft * rng.uniform(1.2, 4.0, N)).round(0)

base = 90_000
price = (base + sqft * 180 * city_multiplier + bedrooms * 8_000 + bathrooms * 12_000
         + garage_spaces * 9_000 + lot_size * 6 - house_age * 1_200) * city_multiplier
price = price * rng.normal(1.0, 0.12, N)       # deliberate 12% noise
price = price.clip(60_000, None).round(-2)

housing = pd.DataFrame({
    'metro': metro, 'location_tier': location_tier, 'sqft': sqft.astype(int),
    'bedrooms': bedrooms, 'bathrooms': bathrooms, 'house_age': house_age,
    'garage_spaces': garage_spaces, 'lot_size': lot_size.astype(int),
    'price': price.astype(int),
})

# The same planted errors as Session 1
bad = rng.choice(N, 12, replace=False)
housing.loc[bad[:6], 'sqft']     = housing.loc[bad[:6], 'sqft'] * 10
housing.loc[bad[6:], 'bedrooms'] = 0

print("Base rebuilt:", housing.shape)
print("Matches S1?  sqft max =", housing.sqft.max(), "| bedrooms==0 count =", (housing.bedrooms == 0).sum())

Base rebuilt: (5000, 9)
Matches S1?  sqft max = 24620 | bedrooms==0 count = 6


### Step 2 — Apply the Session 1 cleaning

The planted errors were the S1 exercise, and you already know the fix: drop the impossible rows. We do it here so every S3 notebook starts from clean ground.

In [3]:
before = len(housing)
housing = housing[(housing.bedrooms > 0) & (housing.sqft <= 10_000)].copy().reset_index(drop=True)
print(f"Dropped {before - len(housing)} bad rows -> {len(housing)} remain.")

Dropped 10 bad rows -> 4990 remain.


### Step 3 — Attach the new columns

Five new fields arrive with the fuller extract. Read the code if you like, but the **data dictionary below the code is the part that matters.**

In [4]:
aug = np.random.default_rng(2026)   # separate seed: base data is untouched
m = len(housing)

# --- Financial records tied to the transaction ---
housing['tax_assessed_value'] = (housing.price * 0.93 * aug.normal(1, 0.035, m)).round(-2).astype(int)
housing['agent_commission']   = (housing.price * 0.027 * aug.normal(1, 0.08, m)).round(0).astype(int)
housing['listing_price']      = (housing.price * aug.normal(1.04, 0.055, m)).round(-2).astype(int)

In [5]:
# --- Transaction date. Prices in this market drifted upward over six years,
#     so date genuinely carries information about price. ---
expected = (housing.sqft * 200 + housing.lot_size * 6 - housing.house_age * 1200
            + (4 - housing.location_tier) * 180_000)
residual = housing.price - expected

order = np.argsort(residual.values + aug.normal(0, residual.std() * 1.45, m))
days  = np.zeros(m, dtype=int)
days[order] = np.sort(aug.integers(0, 2190, m))          # spread across 2019-01-01 .. 2024-12-31
housing['sale_date'] = pd.to_datetime('2019-01-01') + pd.to_timedelta(days, unit='D')

In [6]:
# --- Development identifier. Much of this market is tract housing: builders put up
#     several near-identical units in one development, and they sell at near-identical prices. ---
housing['development_id'] = [f'D{i:05d}' for i in range(m)]

tract = aug.choice(m, 1200, replace=False)      # 1,200 multi-unit developments
siblings = []
for i in tract:
    for _ in range(aug.integers(2, 5)):          # 2-4 sibling units each
        r = housing.loc[i].copy()
        r['sqft']      = int(r['sqft'] * aug.normal(1, 0.010))
        r['house_age'] = int(max(0, r['house_age'] + aug.integers(-1, 2)))
        r['price']     = int(r['price'] * aug.normal(1, 0.008))
        r['tax_assessed_value'] = int(r['price'] * 0.93 * aug.normal(1, 0.035))
        r['agent_commission']   = int(r['price'] * 0.027 * aug.normal(1, 0.08))
        r['listing_price']      = int(r['price'] * aug.normal(1.04, 0.055))
        siblings.append(r)

housing_ext = (pd.concat([housing, pd.DataFrame(siblings)], ignore_index=True)
                 .sample(frac=1, random_state=7)
                 .reset_index(drop=True))

housing_ext.to_csv('housing_extended.csv', index=False)
print(f"Saved housing_extended.csv — {len(housing_ext)} rows, {housing_ext.shape[1]} columns")
print(f"Developments with more than one unit: {(housing_ext.development_id.value_counts() > 1).sum()}")

Saved housing_extended.csv — 8561 rows, 14 columns
Developments with more than one unit: 1200


### Data dictionary — `housing_extended.csv`

**Read the last column.** In a regression problem the model is predicting `price`. Any feature whose value is only determined *after* the price is known cannot be used to predict it — no matter how well it correlates.

| Column | Meaning | **When is this known?** |
|---|---|---|
| `metro` | US city | Before listing |
| `location_tier` | 1 = priciest, 3 = most affordable | Before listing |
| `sqft` | Living area, square feet | Before listing |
| `bedrooms` | Bedroom count | Before listing |
| `bathrooms` | Bathroom count | Before listing |
| `house_age` | Years since built | Before listing |
| `garage_spaces` | Garage capacity, 0–3 | Before listing |
| `lot_size` | Lot area, square feet | Before listing |
| `listing_price` | The asking price the seller advertised | **Before the sale** — set when the property goes on the market |
| `sale_date` | Date the sale completed | At sale |
| `development_id` | Which housing development the unit belongs to | Before listing |
| `tax_assessed_value` | County tax assessment for the property | **After the sale** — the assessor re-values using the recorded sale price |
| `agent_commission` | Commission paid to the selling agent | **After the sale** — it is a percentage of the final price |
| **`price`** | **TARGET** — final sale price, USD | At sale |

> **The trap, stated plainly and then left alone:** three of these columns correlate with `price` above 0.99. Only one of them is safe to use. The data dictionary above contains everything you need to work out which — and Notebook 01 is where you do that work. Do not skip ahead; getting this wrong in the room is more instructive than being told.

In [7]:
# What the new columns look like
housing_ext[['sqft', 'listing_price', 'tax_assessed_value',
             'agent_commission', 'sale_date', 'development_id', 'price']].head(8)

,sqft,listing_price,tax_assessed_value,agent_commission,sale_date,development_id,price
0,840,437100,355000,10877,2021-06-03,D01324,402700
1,1148,375300,315200,9758,2020-02-10,D02830,350900
2,1381,347700,321800,8855,2024-01-28,D03992,357200
3,2603,802600,665100,20249,2022-09-09,D03016,746100
4,2814,2643915,2319864,78434,2024-12-15,D03946,2532358
5,2181,1903000,1588500,47833,2024-04-18,D03853,1831900
6,1489,428500,402800,12554,2020-02-14,D04655,425600
7,1533,324700,295700,8720,2024-07-27,D04963,296100


In [8]:
# Correlation of every numeric column with the target
housing_ext.corr(numeric_only=True)['price'].sort_values(ascending=False).round(3)

price                 1.000
tax_assessed_value    0.998
listing_price         0.995
agent_commission      0.990
sqft                  0.470
bedrooms              0.391
lot_size              0.386
bathrooms             0.335
garage_spaces         0.015
house_age            -0.052
location_tier        -0.724
Name: price, dtype: float64

---

## Part 2 — The subscription churn dataset

### Why a second dataset, and why this one

From Notebook 04 onwards the session needs a **classification** problem. The choice of *which* classification problem is deliberate.

`subscription_churn.csv` describes customers of a fictional broadband and streaming provider. The target is whether the customer **cancelled in the following quarter**.

Structurally, this is the **Employee Turnover CEP** wearing different clothes:

| | Employee Turnover (your CEP) | Subscription churn (today) |
|---|---|---|
| Unit of analysis | One employee | One customer |
| Target | Did they leave? | Did they cancel? |
| Class balance | Minority leave | ~20% cancel |
| Cost of a **false negative** | You lose someone you could have kept | You lose revenue you could have kept |
| Cost of a **false positive** | You spend a retention budget on someone who was staying anyway | Same |
| Deliverable | Risk bands → retention strategy | Risk bands → retention offer |

Same shape, same decision, same metric argument. You will rehearse the reasoning here and then apply it to the graded project. **This is not the CEP dataset and it is not a substitute for it** — it is the practice ground.

In [9]:
# --- Generate the churn dataset ---
c = np.random.default_rng(4242)
M = 12_000

contract = c.choice(['Month-to-month', 'One year', 'Two year'], M, p=[.55, .26, .19])
tenure   = np.where(contract == 'Month-to-month', c.integers(1, 40, M), c.integers(6, 73, M))
monthly  = c.normal(70, 26, M).clip(19, 125).round(2)
total    = (monthly * tenure * c.normal(1, .04, M)).round(2)
tickets  = c.poisson(np.where(contract == 'Month-to-month', 2.1, 1.0), M)
usage    = c.normal(240, 95, M).clip(5, 700).round(1)
late     = c.poisson(0.75, M).clip(0, 9)
premium  = c.choice([0, 1], M, p=[.68, .32])
nserv    = c.integers(1, 7, M)
age      = c.integers(19, 79, M)
region   = c.choice(['Northeast', 'South', 'Midwest', 'West'], M, p=[.22, .31, .23, .24])
satis    = np.clip(np.round(c.normal(7.1, 1.9, M) - tickets * 0.42 - late * 0.30), 1, 10)

In [10]:
# The churn 'signal'. Read this as a story about why people cancel.
z = (-2.45
     + (contract == 'Month-to-month') * 1.30      # no lock-in, easy to walk
     - (contract == 'Two year') * 0.85            # locked in, unlikely to leave
     - 0.030 * tenure                             # long-standing customers stay
     + 0.235 * tickets                            # support problems drive people out
     - 0.265 * (satis - 7)                        # satisfaction is protective
     + 0.175 * late                               # payment friction
     + 0.0105 * (monthly - 70)                    # expensive plans churn more
     - 0.42 * premium                             # premium support helps
     - 0.055 * nserv                              # more services = stickier
     + 0.028 * (monthly - 70) * (7 - satis) * (satis < 6)   # INTERACTION (see note below)
     + 0.85 * ((tenure < 7) & (contract == 'Month-to-month'))  # THRESHOLD (see note below)
    )

p = 1 / (1 + np.exp(-(z + c.normal(0, 0.55, M))))   # unexplainable randomness
churned = (c.random(M) < p).astype(int)

churn = pd.DataFrame({
    'tenure_months': tenure, 'contract_type': contract, 'monthly_charges': monthly,
    'total_charges': total, 'num_support_tickets_6m': tickets,
    'avg_monthly_usage_gb': usage, 'late_payments_12m': late,
    'has_premium_support': premium, 'num_services': nserv, 'age': age,
    'region': region, 'satisfaction_score': satis.astype(int), 'churned': churned,
})
churn.to_csv('subscription_churn.csv', index=False)
print(f"Saved subscription_churn.csv — {len(churn)} rows, churn rate {churn.churned.mean():.1%}")

Saved subscription_churn.csv — 12000 rows, churn rate 19.9%


### Two deliberate design choices worth knowing about

Two lines in the generator above are flagged, and they exist so that **Notebook 05 has something real to test**.

**1. The interaction term.** Customers on expensive plans who are *also* unhappy churn far more than either factor alone would predict. That is a genuine interaction — the effect of price *depends on* satisfaction.

**2. The threshold effect.** Brand-new month-to-month customers (under 7 months) churn at a sharply elevated rate. The jump is abrupt, not gradual.

Why this matters: **logistic regression cannot represent either pattern** without being explicitly told to. It fits one straight-line effect per feature. A tree-based model finds both patterns on its own.

So the two models genuinely disagree about specific customers — which is precisely the situation Notebook 05 needs. Without a designed disagreement, "is model A better than model B?" would have no honest answer to find.

### Data dictionary — `subscription_churn.csv`

| Column | Meaning | Type | Notes |
|---|---|---|---|
| `tenure_months` | Months as a customer | Numeric, 1–72 | Strongly protective |
| `contract_type` | Month-to-month / One year / Two year | Categorical | **Strongest single predictor** |
| `monthly_charges` | Current monthly bill, USD | Numeric, 19–125 | |
| `total_charges` | Lifetime billed, USD | Numeric | ≈ `tenure × monthly` — heavily correlated with both |
| `num_support_tickets_6m` | Support contacts, last 6 months | Numeric, 0–~9 | Strong positive driver |
| `avg_monthly_usage_gb` | Average data usage | Numeric | **Weak** — near-useless by design |
| `late_payments_12m` | Late payments in last year | Numeric, 0–9 | |
| `has_premium_support` | Premium support add-on | Binary | Protective |
| `num_services` | Products subscribed to, 1–6 | Numeric | More services = stickier |
| `age` | Customer age | Numeric, 19–78 | **Weak** — near-useless by design |
| `region` | US region | Categorical | **No real signal** — a decoy |
| `satisfaction_score` | Latest survey score, 1–10 | Numeric | Strong protective driver |
| **`churned`** | **TARGET** — 1 = cancelled next quarter | Binary | ~20% positive |

**Note on the weak columns.** `avg_monthly_usage_gb`, `age` and `region` carry essentially no signal. They are in here on purpose: real datasets are mostly padding, and a model that "finds" a strong `region` effect has found noise. You will use these in Notebook 05's error analysis.

In [11]:
# Sanity check: does the signal behave the way the story says it should?
print("Churn rate by contract type:")
print(churn.groupby('contract_type').churned.mean().round(3).to_string())
print("\nChurn rate by satisfaction score:")
print(churn.groupby('satisfaction_score').churned.mean().round(3).to_string())

Churn rate by contract type:
contract_type
Month-to-month    0.311
One year          0.072
Two year          0.050

Churn rate by satisfaction score:
satisfaction_score
1     0.419
2     0.468
3     0.372
4     0.345
5     0.284
6     0.176
7     0.136
8     0.090
9     0.077
10    0.056


In [12]:
# The number that makes Notebook 04 necessary
majority = 1 - churn.churned.mean()
print(f"Churn rate                      : {churn.churned.mean():.1%}")
print(f"Accuracy of predicting 'nobody churns': {majority:.1%}")
print()
print("A model that never predicts a single cancellation is 80% accurate.")
print("It is also completely worthless. Notebook 04 is about the vocabulary")
print("that lets you say precisely why.")

Churn rate                      : 19.9%
Accuracy of predicting 'nobody churns': 80.2%

A model that never predicts a single cancellation is 80% accurate.
It is also completely worthless. Notebook 04 is about the vocabulary
that lets you say precisely why.


---

## What you now have

Two files sit next to this notebook:

| File | Rows | Used by | Target |
|---|---|---|---|
| `housing_extended.csv` | ~8,600 | Notebooks 01, 02, 03 | `price` (regression) |
| `subscription_churn.csv` | 12,000 | Notebooks 04, 05 | `churned` (classification) |

Both are **synthetic**. That has one honest cost worth stating: synthetic data is cleaner and better-behaved than reality, so everything here will work slightly more neatly than it will on the job. The compensating benefit is that we chose the problems on purpose — every trap you meet today is one we planted, which means every trap is one you can fully understand rather than merely survive.

Real mess arrives in the CEP clinics.

---

**Next:** open `ML_S3_01_splitting_and_leakage.ipynb`.